# Task 13: frozen Jais pilot
Use a fresh Colab GPU runtime. Run setup and the 20-clip smoke first; inspect its
results before the full 500/200 pilot. Nothing in this notebook trains Jais.
Outputs and caches are saved to your existing data directory on Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU.'
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
assert DATA_ROOT.is_dir(), f'Missing data directory: {DATA_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Data directory:', DATA_ROOT)

In [ ]:
import subprocess, sys, tempfile, json
from datetime import datetime, timezone

SOURCE_REF = 'eec-17-jais-pilot'
REPO_ROOT = Path(tempfile.mkdtemp(prefix='jais-task13-'))
subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', SOURCE_REF,
                'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git', str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_ROOT / 'environments/jais.txt')], check=True)
# Installation uses observed Jais dependencies, retaining the runtime's Torch build.
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
RUN_ROOT = DATA_ROOT / 'runs/jais_task13' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
AUDIO_ROOT = DATA_ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'
JAIS_CACHE = DATA_ROOT / 'cache/jais_adapter_module_v1'
HF_CACHE = Path('/content/hf-jais-task13')
assert AUDIO_ROOT.is_dir(), f'Missing prepared audio: {AUDIO_ROOT}'
print('Source:', SOURCE_COMMIT)
print('Outputs:', RUN_ROOT)

def command(script, args, allowed=(0,)):
    result = subprocess.run([sys.executable, str(REPO_ROOT / script), *map(str, args)], cwd=REPO_ROOT)
    if result.returncode not in allowed:
        raise RuntimeError(f'{script} exited {result.returncode}; inspect output above')
    return result.returncode

In [ ]:
# Deterministic 20-clip smoke: one frozen training clip per dialect.
SMOKE_INPUT = RUN_ROOT / 'smoke-input'
command('src/experiments/jais_pilot.py', ['prepare-smoke',
    '--manifest', REPO_ROOT / 'data/manifests/train_pilot.jsonl',
    '--metadata', REPO_ROOT / 'data/manifests/train_pilot.metadata.json',
    '--output-dir', SMOKE_INPUT])
smoke_rows = [json.loads(line) for line in (SMOKE_INPUT / 'smoke.jsonl').read_text().splitlines()]
missing = [r['clip_id'] for r in smoke_rows if not (AUDIO_ROOT / r['audio_path']).is_file()]
assert not missing, f'{len(missing)} smoke WAV files are missing; check AUDIO_ROOT first'
print('Smoke clips:', len(smoke_rows), 'All 20 WAV files found.')

In [ ]:
# GPU smoke. Inspect this export before running the remaining cells.
def run_split(name, manifest, metadata, split):
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = RUN_ROOT / f'{name}-{attempt}'
    status = command('src/models/jais_adapter.py', [
        '--manifest', manifest, '--metadata', metadata, '--audio-root', AUDIO_ROOT,
        '--cache-root', JAIS_CACHE, '--model-cache', HF_CACHE, '--output-dir', out,
        '--dtype', 'float16', '--batch-size', '2', '--max-new-tokens', '128'], allowed=(0, 2))
    exported = RUN_ROOT / f'{name}-export-{attempt}'
    command('src/experiments/jais_pilot.py', ['export', '--manifest', manifest,
        '--metadata', metadata, '--split', split, '--run-dir', out, '--output-dir', exported])
    command('src/evaluation/evaluate.py', ['--manifest', manifest, '--metadata', metadata,
        '--predictions', out / 'predictions.jsonl', '--labels', REPO_ROOT / 'configs/labels.json',
        '--split', split, '--model-name', 'jais-direct-pilot', '--output-dir', exported / 'evaluation'])
    summary = json.loads((exported / 'summary.json').read_text())
    print('Adapter exit:', status)
    print('Transcript status counts:', summary['transcript_counts'])
    print('Resource counts:', summary['resource_record']['counts'])
    print('Refused / truncated / unknown:', summary['refused'], summary['truncated'], summary['truncation_unknown'])
    print('Export:', exported)
    return out, exported

SMOKE_RUN, SMOKE_EXPORT = run_split('smoke', SMOKE_INPUT / 'smoke.jsonl',
                                   SMOKE_INPUT / 'smoke.metadata.json', 'train')

## Inspect smoke before continuing
Keep all outputs, including failures. Check mapping errors, refusals, truncation
and resource counts. If generation settings need changing, choose them before
running both full subsets and use the same settings for train and dev.
A 20-clip smoke accuracy is not a benchmark result.

In [ ]:
TRAIN_RUN, TRAIN_EXPORT = run_split('train', REPO_ROOT / 'data/manifests/train_pilot.jsonl',
                                   REPO_ROOT / 'data/manifests/train_pilot.metadata.json', 'train')

In [ ]:
DEV_RUN, DEV_EXPORT = run_split('dev', REPO_ROOT / 'data/manifests/dev_pilot.jsonl',
                               REPO_ROOT / 'data/manifests/dev_pilot.metadata.json', 'dev')
metrics = json.loads((DEV_EXPORT / 'evaluation/metrics.json').read_text())
print('Development accuracy:', metrics['accuracy'])
print('Development macro-F1:', metrics['macro_f1'])
print('Both full pilot exports:', TRAIN_EXPORT, DEV_EXPORT)